# Week 3: Exploratory Data Analysis & Feature Engineering
## Notebook 09: UMAP: Uniform Manifold Approximation and Projection

### 1. Conceptual Intuition & Theoretical Background
**UMAP** is a non-linear dimension reduction algorithm grounded in Riemannian geometry and algebraic topology. Unlike t-SNE (which focuses almost exclusively on local neighborhoods), UMAP preserves both **local clusters and global continuum structures**.

UMAP operates in two phases:
1. **High-dimensional graph construction:** Builds a fuzzy simplicial set where edge weights represent the probability that points are connected on a Riemannian manifold:
$$p_{i|j} = \exp\left( -\frac{\max(0, d(x_i, x_j) - \rho_i)}{\sigma_i} \right)$$
Where $\rho_i$ is distance to the nearest neighbor (ensuring local connectivity), and $\sigma_i$ is a local scale factor solved via binary search.

2. **Low-dimensional layout optimization:** Embeds points into 2D/3D space by minimizing the **Fuzzy Set Cross-Entropy**:
$$\mathcal{L} = \sum_{e} \left[ p_e \log \frac{p_e}{q_e} + (1 - p_e) \log \frac{1 - p_e}{1 - q_e} \right]$$
Where $q_{ij} = (1 + a \|y_i - y_j\|^{2b})^{-1}$.

---

### 2. Scratch Python Implementation: High-Dimensional Local Metric

In [ ]:
import numpy as np

# 1. Compute Pairwise Euclidean Distances
def compute_distance_matrix(X):
    sum_X = np.sum(X**2, axis=1, keepdims=True)
    D2 = sum_X + sum_X.T - 2 * np.dot(X, X.T)
    return np.sqrt(np.maximum(D2, 0.0))

# 2. Binary search for local scale sigma_i such that sum of probabilities equals log2(k)
def find_local_sigmas(D, k=15, max_iter=50):
    n = D.shape[0]
    target = np.log2(k)
    rhos = np.zeros(n)
    sigmas = np.ones(n)
    
    for i in range(n):
        row = D[i].copy()
        row[i] = np.inf
        rhos[i] = np.min(row)  # Distance to 1st nearest neighbor
        
        # Binary search for sigma
        low = 1e-5
        high = 100.0
        for _ in range(max_iter):
            mid = (low + high) / 2.0
            val = np.sum(np.exp(-np.maximum(0.0, D[i] - rhos[i]) / mid)) - 1.0  # exclude self
            if val > target:
                high = mid
            else:
                low = mid
        sigmas[i] = mid
        
    return rhos, sigmas

# 3. Symmetrized Fuzzy Simplicial Graph Construction
def build_fuzzy_simplicial_graph(X, k=10):
    D = compute_distance_matrix(X)
    rhos, sigmas = find_local_sigmas(D, k=k)
    
    # Directed probabilities
    P = np.exp(-np.maximum(0.0, D - rhos[:, None]) / sigmas[:, None])
    np.fill_diagonal(P, 0.0)
    
    # Fuzzy union: P_sym = P + P.T - P * P.T
    P_sym = P + P.T - (P * P.T)
    return P_sym

# Create 3 synthetic 3D clusters
np.random.seed(42)
c1 = np.random.normal(loc=[-5, 0, 0], scale=0.5, size=(40, 3))
c2 = np.random.normal(loc=[0, 5, 0], scale=0.5, size=(40, 3))
c3 = np.random.normal(loc=[5, 0, 0], scale=0.5, size=(40, 3))
X_toy = np.vstack([c1, c2, c3])

P_graph = build_fuzzy_simplicial_graph(X_toy, k=8)
print(f"Graph Adjacency Shape: {P_graph.shape}")
print(f"Mean Non-Zero Edge Probability: {P_graph[P_graph > 0.01].mean():.4f}")
print("Fuzzy simplicial topology constructed successfully!")

### 3. Step-by-Step Code Walkthrough
1. **Local Connectivity ($\rho_i$)**: Ensures each point is connected to its nearest neighbor with probability 1.0, preserving isolated clusters.
2. **Adaptive Bandwidth ($\sigma_i$)**: Adjusts local scale depending on neighborhood density.
3. **Fuzzy Union**: Symmetrizes probabilities so $p_{ij} = p_{ji}$ via algebraic t-conorm.

---

### 4. Industry Applications & Trade-offs
- **Single-cell RNA sequencing (scRNA-seq):** Visualizing continuous cellular differentiation trajectories.
- **Trade-off:** Sensitive to initialization; UMAP creates visually appealing clusters even on uniform random noise if $n\_neighbors$ is too small.

---

### 5. Top Interview Questions & Answers
**Q1: How does UMAP mathematically preserve global structure better than t-SNE?**
> **Answer:** t-SNE uses standard Kullback-Leibler (KL) divergence, which heavily penalizes placing distant points close together but barely penalizes placing close points far apart. UMAP uses Fuzzy Cross-Entropy, which penalizes both types of placement errors equally.

**Q2: What is the role of the `min_dist` parameter in UMAP?**
> **Answer:** `min_dist` controls how tightly UMAP packs points in the low-dimensional embedding. A small value creates tight cluster clumps, while a larger value preserves broader topological layout.